# CP3: fine-tuning the students on Kaggle (Qwen3 + LoRA)

**Settings:** Accelerator **GPU T4 x2** (one GPU is used), **Internet on**.

**Recommended way to run the grid: Save Version → "Save & Run All (Commit)".** It runs headless for up to
12 hours, so you can close the tab. With `MODE = "grid"` (next cell) it skips the smoke test and runs every
grid run that has no test predictions yet, then zips the results. Download `results_cp3.zip` from the
version's **Output** tab when it finishes, unzip it in the repo root, commit and push.

Before each run: `git push` from your Mac, so the notebook clones the latest code and `results/results.csv`.


In [ ]:
MODE = "grid"   # "smoke" = H and T at 551, seed 0, dev only (CP3a); "grid" = all remaining runs (CP3b)

In [ ]:
import os, subprocess
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Zografska/greenwashing-apprentice.git",
                    "/kaggle/working/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "--ff-only"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Install Unsloth (4-bit QLoRA). If it fails, fall back to transformers + PEFT + bitsandbytes.
!pip install -q unsloth
ok = subprocess.run(["python", "-c", "import unsloth"], capture_output=True, text=True)
if ok.returncode == 0:
    BACKEND = "unsloth"
else:
    print(ok.stderr[-2000:])
    !pip install -q peft bitsandbytes
    BACKEND = "hf"
!pip install -q scikit-learn scipy openpyxl pytest sentence-transformers
print("BACKEND =", BACKEND)
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU', torch.cuda.get_device_name(0))"


In [ ]:
!python -m pytest -q tests
!python -m apprentice.check_split | tail -3

## Session 1: smoke test (CP3a), dev only

In [ ]:
if MODE == "smoke":
    !python -m apprentice.train --condition H --n 551 --seed 0 --backend {BACKEND} --tag smoke
    !python -m apprentice.train --condition T --n 551 --seed 0 --backend {BACKEND} --tag smoke
else:
    print("MODE = grid: smoke test skipped")

## Grid sessions (CP3b): dev + test, once per run

Runs in priority order (the minimum viable project first). Each run trains, scores dev and test, appends to
`results/results.csv`, and is skipped next time once its test predictions exist. Stop any time; rerun the
cell in the next session to continue. Adapters are saved for T-4B and T-1.7B at n=551, seed 0 (needed for the
GGUF export in CP4).

In [ ]:
import itertools, pathlib, time
if MODE != "grid":
    raise SystemExit("MODE is not grid")
RUNS = []
RUNS += [("H", 551, s) for s in (0, 1, 2)] + [("T", 551, s) for s in (0, 1, 2)]          # core RQ1
RUNS += [("T+R", 551, s) for s in (0, 1, 2)]                                               # RQ5
RUNS += [(c, n, s) for n in (300, 150) for s in (0, 1, 2) for c in ("H", "T")]             # learning curve
RUNS += [("T-large", 551, s) for s in (0, 1, 2)]
RUNS += [("T-1.7B", 551, s) for s in (0, 1, 2)]                                            # size ablation
def done(c, n, s):
    tag = f"n{'all' if c == 'T-large' else n}-s{s}"
    return pathlib.Path(f"results/preds/{c}/{tag}/test.jsonl").exists()
todo = [r for r in RUNS if not done(*r)]
print(f"{len(RUNS) - len(todo)} done, {len(todo)} to go")
for c, n, s in todo:
    t0 = time.time()
    extra = ["--save-adapter"] if (c in ("T", "T-1.7B") and n == 551 and s == 0) else []
    r = subprocess.run(["python", "-m", "apprentice.train", "--condition", c, "--n", str(n), "--seed", str(s),
                        "--backend", BACKEND, "--score-test", *extra], capture_output=True, text=True)
    tail = [l for l in r.stdout.splitlines() if "macro-F1" in l or "trained:" in l or "valid JSON" in l]
    print(f"{c} n={n} s={s}  ({(time.time() - t0) / 60:.1f} min, exit {r.returncode})", *tail, sep="\n  ")
    if r.returncode != 0:
        print(r.stderr[-3000:]); break

## Download

In [ ]:
!zip -qr /kaggle/working/results_cp3.zip results/preds results/results.csv runs -x "runs/*/train.jsonl"
print("Download /kaggle/working/results_cp3.zip, unzip in the repo root, commit, push.")